# ORION Experiments (Dense vs Window vs Sparse)

This notebook is the **main comparison workflow**.

- It compares Dense, Window, and Sparse using config profiles.
- Sparse norm control is fixed to the winner from tuning: **none** (`qk_norm=False`, `ortho_init=False`, `spectral_norm=False`).

Use `sparse_norm_control_tuning.ipynb` only when you want to retune norm controls.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/akashkguw/orion.git"

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/orion")
    REPO_ROOT = DRIVE_ROOT / "repo"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").exists()


def _clone_repo(repo_url: str, repo_root: Path, retries: int = 3) -> None:
    last_error = "unknown"
    for attempt in range(1, retries + 1):
        proc = subprocess.run(
            ["git", "clone", repo_url, str(repo_root)],
            text=True,
            capture_output=True,
        )
        if proc.returncode == 0:
            return

        last_error = (proc.stderr or proc.stdout or "clone failed").strip()
        tail = last_error.splitlines()[-1] if last_error else "clone failed"
        print(f"Clone attempt {attempt}/{retries} failed: {tail}")

        if repo_root.exists() and not _is_git_repo(repo_root):
            shutil.rmtree(repo_root, ignore_errors=True)

        if attempt < retries:
            time.sleep(attempt)

    raise RuntimeError(f"Failed to clone repository after {retries} attempts: {last_error}")


if REPO_ROOT.exists() and not _is_git_repo(REPO_ROOT):
    if any(REPO_ROOT.iterdir()):
        backup = REPO_ROOT.with_name(f"{REPO_ROOT.name}_backup_{int(time.time())}")
        print(f"Found non-git directory at {REPO_ROOT}; moving to {backup}")
        REPO_ROOT.rename(backup)
    else:
        REPO_ROOT.rmdir()

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    _clone_repo(REPO_URL, REPO_ROOT)
else:
    print(f"Using existing repository at {REPO_ROOT}")

os.chdir(REPO_ROOT)
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
if IN_COLAB and USE_GOOGLE_DRIVE:
    print(f"Google Drive persistence enabled at: {REPO_ROOT}")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dev.txt",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "seaborn"], check=True)
print("Dependencies installed.")

In [ ]:
from pathlib import Path

import seaborn as sns

from orion.experiments import (
    load_profile_context,
    load_summary_df,
    paired_analysis_tables,
    plot_all_numeric_metrics,
    plot_speedup_ratios,
    plot_vram_and_quality,
    prepare_analysis_columns,
    run_profile,
    select_winners,
)

sns.set_theme(style="whitegrid")

# Main comparison profiles (no norm-control sweep here).
PROFILE = "pilot9"  # pilot9 | pilot | full | w64_d_sweep
OVERWRITE = False
ENFORCE_SPARSE_NORM_WINNER_NONE = True

PROFILE_CTX = load_profile_context(PROFILE)
PROFILE_CTX

In [ ]:
if PROFILE == "pilot_norm":
    raise ValueError(
        "This notebook is for main Dense/Window/Sparse comparison. "
        "Use sparse_norm_control_tuning.ipynb for norm-control sweeps."
    )

run_result = run_profile(PROFILE, overwrite=OVERWRITE)
run_result

In [ ]:
df = load_summary_df(run_result.summary_csv)
df_ok = prepare_analysis_columns(df[df["status"] == "ok"].copy())

print("Successful trials:", len(df_ok), "/", len(df))
display(df_ok.head(20))

In [ ]:
# Enforce sparse norm-control winner from tuning: none
sparse_rows = df_ok[df_ok["backend"] == "sparse"].copy()
for col in ["qk_norm", "ortho_init", "spectral_norm"]:
    if col not in sparse_rows.columns:
        sparse_rows[col] = False

norm_summary = (
    sparse_rows[["qk_norm", "ortho_init", "spectral_norm"]]
    .fillna(False)
    .astype(bool)
    .value_counts(dropna=False)
    .reset_index(name="count")
)
print("Sparse norm-control modes observed in this run:")
display(norm_summary)

if ENFORCE_SPARSE_NORM_WINNER_NONE:
    bad = sparse_rows[
        (sparse_rows["qk_norm"]) | (sparse_rows["ortho_init"]) | (sparse_rows["spectral_norm"])
    ]
    if len(bad) > 0:
        raise ValueError(
            "Found sparse runs with non-winning norm control. "
            "Expected qk_norm=False, ortho_init=False, spectral_norm=False."
        )

print("Sparse norm control check passed.")

In [ ]:
paired_all, paired, agg = paired_analysis_tables(df_ok)

print(f"Paired rows (all non-dense variants): {len(paired_all)}")
print(f"Paired rows (focused: window + best sparse per key): {len(paired)}")
display(agg.sort_values(["seq_len", "backend", "sparse_tag", "lr"]))

winners = select_winners(agg, val_ppl_tolerance=PROFILE_CTX.val_ppl_tolerance)
print("Candidate wins over dense:")
display(winners)

In [ ]:
plot_speedup_ratios(paired)
plot_vram_and_quality(paired)

In [ ]:
plots_dir = Path(run_result.runs_root) / "plots_all_metrics"
metric_cols = plot_all_numeric_metrics(df_ok, out_dir=plots_dir, show_inline=True)
display(metric_cols)
print("Saved plots to:", plots_dir)